# Phase 1: Scale-Up and Re-Labeling (USAspending CSV)

**Purpose:** Re-run the filtering and label construction pipeline on USAspending CSV downloads.

**Inputs:** `C:/Users/sarme/IS392Final/data/*.csv` (from zip download)
**Outputs:**
- `data/interim/filtered_physical_deliverables.csv`
- `data/processed/labeled_contracts.csv`
- Updated `data/processed/dataset_summary.txt`

**Key Design Decision:** PIID-group sampling preserves complete contract histories by selecting PIIDs first, then keeping all rows (modifications) for those contracts.

**Changes from Parquet version:**
- CSV files instead of Parquet shards
- Human-readable USAspending column names
- Pre-filtered on download (sanity checks only)


## 1. Environment Setup and Imports

In [6]:
# Data handling
import pandas as pd
import numpy as np
import zipfile
import os
import glob
import warnings
from pathlib import Path
from datetime import datetime

# Progress bar
from tqdm import tqdm

# Suppress noisy warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

# Global reproducibility seed
RANDOM_STATE = 42

print('Environment configured. Random state:', RANDOM_STATE)

Environment configured. Random state: 42


## 2. Configuration

All paths, filters, and thresholds defined as constants for reproducibility.

In [ ]:
# --- File Paths ---
import os
from pathlib import Path

print('=' * 70)
print('STEP 2: CONFIGURATION')
print('=' * 70)

# Define data directory - contains parquet and CSV files from USAspending
DATA_DIR = Path(r'C:/Users/sarme/IS392Final/data').resolve()

# Verify data directory exists
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Data directory not found: {DATA_DIR}')

print(f'\n✓ Data directory verified: {DATA_DIR}')

# Define CSV_FOLDER - primary path for loading data files
CSV_FOLDER = str(DATA_DIR)
print(f'✓ CSV_FOLDER defined: {CSV_FOLDER}')

# Verify CSV_FOLDER has data files
csv_count = len(list(Path(CSV_FOLDER).glob('*.csv')))
parquet_count = len(list(Path(CSV_FOLDER).glob('*.parquet')))
print(f'  Found: {csv_count} CSV files, {parquet_count} Parquet files')

ZIP_FILE = None  # Set to path if CSVs are in a zip

# Intermediate checkpoint after filtering
INTERIM_OUTPUT = str(DATA_DIR / 'interim' / 'filtered_physical_deliverables.csv')
# Final labeled dataset output
FINAL_OUTPUT = str(DATA_DIR / 'processed' / 'labeled_contracts.csv')
# Dataset summary text file
SUMMARY_FILE = str(DATA_DIR / 'processed' / 'dataset_summary.txt')

# Create output directories if they do not exist
os.makedirs(os.path.dirname(INTERIM_OUTPUT), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_OUTPUT), exist_ok=True)

# --- Filtering Criteria ---
# NOTE: Data is pre-filtered on USAspending download ($500K+, target PSC/NAICS)
# These filters act as sanity checks
PHYSICAL_PSC_PREFIXES = ['Y', 'Z']
PHYSICAL_PSC_NUMERIC_RANGE = (10, 99)
MIN_CONTRACT_VALUE = 500_000  # Sanity check - should already be applied

# --- Labeling Thresholds ---
# Primary cost overrun threshold: 5% growth triggers over_budget=1
COST_OVERRUN_THRESHOLD = 0.05
# Adaptive fallback threshold if positive class is too small (<5%)
COST_OVERRUN_THRESHOLD_FALLBACK = 0.01
# Any positive delay in days triggers late=1
SCHEDULE_DELAY_THRESHOLD = 0

# --- Sampling Configuration ---
# Target number of unique PIID contracts to sample (None = use all)
SAMPLE_CONTRACTS = None  # Set to integer (e.g., 50000) to sample, None to use all

# --- USAspending Column Mapping ---
# Maps semantic names to USAspending CSV column names
COLUMN_MAP = {
    'piid': 'award_id_piid',
    'mod_number': 'modification_number',
    'description': 'transaction_description',
    'psc': 'product_or_service_code',
    'naics': 'naics_code',
    'base_all_options': 'potential_total_value_of_award',
    'base_exercised': 'current_total_value_of_award',
    'obligated_amount': 'federal_action_obligation',
    'current_completion': 'period_of_performance_current_end_date',
    'ultimate_completion': 'period_of_performance_potential_end_date',
    'effective_date': 'period_of_performance_start_date',
    'signed_date': 'action_date',
    'contract_type': 'type_of_contract_pricing',
    'extent_competed': 'extent_competed',
    'num_offers': 'number_of_offers_received',
    'agency_id': 'awarding_agency_code',
    'vendor_name': 'recipient_name',
    # Note: state_code column may not be available in all USAspending downloads
    # 'state_code': 'place_of_performance_state',
    'mod_reason': 'action_type',
}

# List of columns to read (subset for efficiency)
COLUMNS_TO_READ = list(COLUMN_MAP.values())

print('\n✓ Configuration defined:')
print(f'  - COLUMN_MAP: {len(COLUMN_MAP)} mappings')
print(f'  - COLUMNS_TO_READ: {len(COLUMNS_TO_READ)} columns to load')
print(f'  - MIN_CONTRACT_VALUE: ${MIN_CONTRACT_VALUE:,}')
print(f'  - SAMPLE_CONTRACTS: {"All PIIDs" if SAMPLE_CONTRACTS is None else f"{SAMPLE_CONTRACTS:,} PIIDs"}')
print(f'  - COST_OVERRUN_THRESHOLD: {COST_OVERRUN_THRESHOLD:.0%}')
print(f'  - SCHEDULE_DELAY_THRESHOLD: {SCHEDULE_DELAY_THRESHOLD} days')
print('\n✓ All configuration variables ready')

Configuration loaded.
  CSV folder: C:\Users\sarme\IS392Final\data
  Columns mapped: 18
  Sampling: All PIIDs


## 3. Helper Functions

Reusable functions for filtering contracts and computing labels.

In [ ]:
def is_physical_deliverable(psc_code: str) -> bool:
    """
    Check if a PSC code indicates a physical deliverable.
    
    Parameters
    ----------
    psc_code : str
        The product or service code to check.
    
    Returns
    -------
    bool
        True if PSC indicates construction (Y), maintenance (Z),
        or supplies/equipment (10-99 numeric range).
    """
    if pd.isna(psc_code):
        return False
    psc_str = str(psc_code).strip().upper()
    # Check for Y or Z prefix (construction or maintenance)
    if psc_str.startswith(('Y', 'Z')):
        return True
    # Check for numeric range 10-99 (supplies/equipment)
    try:
        psc_num = int(psc_str)
        return PHYSICAL_PSC_NUMERIC_RANGE[0] <= psc_num <= PHYSICAL_PSC_NUMERIC_RANGE[1]
    except (ValueError, IndexError):
        return False


def compute_cost_growth(base_val, final_val) -> float:
    """
    Compute percentage cost growth between base and final contract values.
    
    Parameters
    ----------
    base_val : numeric or str
        Initial contract value (potential_total_value_of_award).
    final_val : numeric or str
        Final contract value (current_total_value_of_award).
    
    Returns
    -------
    float
        Percentage growth as decimal (e.g., 0.05 = 5% growth).
        Returns np.nan if values are invalid or base is zero.
    """
    try:
        # Clean string values (remove $ and commas)
        base = float(str(base_val).replace('$', '').replace(',', '').strip())
        final = float(str(final_val).replace('$', '').replace(',', '').strip())
        if base == 0:
            return np.nan
        return (final - base) / abs(base)
    except (ValueError, TypeError):
        return np.nan


def compute_delay(current_date, ultimate_date) -> float:
    """
    Compute schedule delay in days between current and ultimate completion dates.
    
    Parameters
    ----------
    current_date : str or datetime
        Original/current completion date.
    ultimate_date : str or datetime
        Ultimate (final) completion date.
    
    Returns
    -------
    float
        Delay in days (positive = delayed, negative = early).
        Returns np.nan if dates are invalid.
    """
    try:
        current = pd.to_datetime(current_date)
        ultimate = pd.to_datetime(ultimate_date)
        return (ultimate - current).days
    except (ValueError, TypeError):
        return np.nan


print('Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay')

Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay


## 4. Data Loading and Schema Discovery

Load CSV files from zip or folder, concatenate into single DataFrame.

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

import pandas as pd

try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

print('=' * 70)
print('STEP 4: DATA LOADING AND SCHEMA DISCOVERY')
print('=' * 70)

# ===========================================================================
# CRITICAL: Define or import variables from Configuration cell
# ===========================================================================

# Get CSV_FOLDER from config or define it
if 'CSV_FOLDER' not in globals():
    CSV_FOLDER = r'C:/Users/sarme/IS392Final/data'
    print(f'\n⚠ CSV_FOLDER not from config, using: {CSV_FOLDER}')
else:
    print(f'\n✓ CSV_FOLDER from config: {CSV_FOLDER}')

# Get ZIP_FILE from config or define it
if 'ZIP_FILE' not in globals():
    ZIP_FILE = None
    print('⚠ ZIP_FILE not defined')

# Get COLUMNS_TO_READ from config or define it
if 'COLUMNS_TO_READ' not in globals():
    COLUMNS_TO_READ = None
    print('⚠ COLUMNS_TO_READ not defined (will load all columns)')
else:
    print(f'✓ COLUMNS_TO_READ defined: {len(COLUMNS_TO_READ)} columns')

# CRITICAL: Get COLUMN_MAP from config or define it
if 'COLUMN_MAP' not in globals() or not COLUMN_MAP:
    print('\n⚠ ERROR: COLUMN_MAP not defined or empty!')
    print('COLUMN_MAP must be defined in Configuration cell.')
    COLUMN_MAP = {}
else:
    print(f'✓ COLUMN_MAP from config: {len(COLUMN_MAP)} mappings')

# ===========================================================================
# Data Loading Function
# ===========================================================================

def load_usaspending_data(csv_folder, zip_file=None, columns_to_read=None):
    """
    Load USAspending data from CSV or Parquet files from folder or zip file.
    
    Parameters
    ----------
    csv_folder : str
        Path to folder containing CSV or Parquet files
    zip_file : str or None
        Path to zip file containing CSV files (if applicable)
    columns_to_read : list or None
        List of column names to read (if None, reads all columns)
    
    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame from all data files
    """
    # Ensure folder exists
    folder_path = Path(csv_folder)
    if not folder_path.exists():
        raise FileNotFoundError(f'Data folder not found: {csv_folder}')
    
    # Extract from zip if provided
    if zip_file and Path(zip_file).exists():
        print(f'Extracting from zip: {zip_file}')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(csv_folder)
        print(f'Extracted to: {csv_folder}')
    
    # Find all CSV and Parquet files
    csv_files = sorted(glob.glob(os.path.join(csv_folder, '*.csv')))
    parquet_files = sorted(glob.glob(os.path.join(csv_folder, '*.parquet')))

    # Prefer CSV exports when present; parquet is fallback only.
    if csv_files:
        all_files = csv_files
        if parquet_files:
            print(f'Found {len(csv_files)} CSV files and {len(parquet_files)} Parquet files')
            print('Using CSV files only (parquet ignored to avoid mixed-schema loads).')
    else:
        all_files = parquet_files
        if parquet_files:
            print(f'No CSV files found. Using {len(parquet_files)} Parquet files.')
    
    if not all_files:
        raise FileNotFoundError(
            f'No CSV or Parquet files found in {csv_folder}. '
            f'Verify CSV_FOLDER points to the correct location.'
        )

    if all_files and all_files[0].endswith('.parquet') and pq is None:
        raise ImportError(
            'Parquet files were selected but pyarrow is not installed. '
            'Install pyarrow or provide CSV files in CSV_FOLDER.'
        )
    
    # Load and concatenate all files
    dfs = []
    
    for i, file_path in enumerate(all_files, 1):
        filename = os.path.basename(file_path)
        print(f'  Loading {i}/{len(all_files)}: {filename}...', end=' ')
        try:
            if file_path.endswith('.csv'):
                # Read selected columns if available; otherwise load all columns.
                try:
                    if columns_to_read:
                        df = pd.read_csv(file_path, usecols=columns_to_read, low_memory=False)
                    else:
                        df = pd.read_csv(file_path, low_memory=False)
                except (ValueError, KeyError):
                    # If column selection fails, read all columns
                    df = pd.read_csv(file_path, low_memory=False)
            elif file_path.endswith('.parquet'):
                # Read parquet file
                try:
                    if columns_to_read:
                        table = pq.read_table(file_path, columns=columns_to_read)
                    else:
                        table = pq.read_table(file_path)
                    df = table.to_pandas()
                except (ValueError, KeyError):
                    # If column selection fails, read all columns
                    table = pq.read_table(file_path)
                    df = table.to_pandas()
            else:
                print('SKIPPED: unsupported extension')
                continue
            
            dfs.append(df)
            print(f'{len(df):,} rows')
        except Exception as e:
            print(f'ERROR: {e}')
            continue
    
    if not dfs:
        raise ValueError('No data could be loaded from available files')
    
    # Concatenate
    combined_df = pd.concat(dfs, ignore_index=True)
    
    print(f'\nTotal loaded: {len(combined_df):,} rows')
    print(f'Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
    
    return combined_df


# ===========================================================================
# Execute Data Loading
# ===========================================================================

print(f'\nLoading data from: {CSV_FOLDER}')
df = load_usaspending_data(CSV_FOLDER, ZIP_FILE, COLUMNS_TO_READ)

# Show column info
print(f'\nColumns in dataset: {len(df.columns)} columns')
print(f'Rows in dataset: {len(df):,} rows')

# Validate mapped columns
if COLUMN_MAP and len(COLUMN_MAP) > 0:
    print('\nValidating mapped columns:')
    missing = []
    found = 0
    for semantic, actual in COLUMN_MAP.items():
        if actual not in df.columns:
            missing.append(f'  - {semantic}: {actual}')
        else:
            found += 1
    
    print(f'  Found: {found}/{len(COLUMN_MAP)} mapped columns')
    if missing:
        print('  Missing columns:')
        for msg in missing:
            print(msg)
    else:
        print('  ✓ All mapped columns found')
else:
    print('\n⚠ WARNING: COLUMN_MAP is empty or not defined!')
    print('This will cause errors in subsequent cells.')

print('\n✓ Data loading complete')

⚠ CSV_FOLDER was not defined; using fallback: C:/Users/sarme/IS392Final/data
⚠ COLUMNS_TO_READ not defined; will load all columns
⚠ ERROR: COLUMN_MAP not defined or empty. Run Configuration cell first!

Loading data from: C:/Users/sarme/IS392Final/data
Found 1 CSV files and 589 Parquet files
Using CSV files only (parquet ignored to avoid mixed-schema loads).
  Loading 1/1: fpds_subsample.csv... 

## 5. Filtering to Physical Deliverables

Sanity checks - data should already be pre-filtered on USAspending download.

In [ ]:
# Ensure required imports exist
if 'pd' not in globals():
    import pandas as pd
if 'Path' not in globals():
    from pathlib import Path
if 'datetime' not in globals():
    from datetime import datetime
if 'np' not in globals():
    import numpy as np
if 'tqdm' not in globals():
    from tqdm import tqdm

# Verify required data exists
if 'df' not in globals():
    raise NameError('df not defined. Run cells 1-4 (Configuration, Helpers, Data Loading) in order.')

if 'COLUMN_MAP' not in globals() or not COLUMN_MAP:
    raise KeyError('COLUMN_MAP not defined or empty. Run Configuration cell first.')

if 'psc' not in COLUMN_MAP:
    raise KeyError("COLUMN_MAP['psc'] required. Check Configuration cell.")

if 'INTERIM_OUTPUT' not in globals():
    from pathlib import Path
    _data_dir = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    INTERIM_OUTPUT = str(_data_dir / 'interim' / 'filtered_physical_deliverables.csv')

if 'MIN_CONTRACT_VALUE' not in globals():
    MIN_CONTRACT_VALUE = 500_000

# Define helper function if not already defined
if 'is_physical_deliverable' not in globals():
    def is_physical_deliverable(psc_code: str) -> bool:
        if pd.isna(psc_code):
            return False
        psc_str = str(psc_code).strip().upper()
        if psc_str.startswith(('Y', 'Z')):
            return True
        try:
            psc_num = int(psc_str)
            return 10 <= psc_num <= 99
        except (ValueError, TypeError):
            return False

# Initialize metadata tracker
run_metadata = {
    'run_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_rows_loaded': len(df),
    'total_rows_after_psc_check': 0,
    'total_rows_after_value_check': 0,
    'unique_piids_before_sample': 0,
    'unique_piids_after_sample': 0,
    'final_labeled_count': 0,
}

# Apply PSC sanity check
print('Applying PSC sanity check...')
psc_col = COLUMN_MAP['psc']
if psc_col not in df.columns:
    raise KeyError(f"Required PSC column '{psc_col}' not found in dataset.")

physical_mask = df[psc_col].apply(is_physical_deliverable)
df_filtered = df[physical_mask].copy()

psc_removed = len(df) - len(df_filtered)
psc_pct = (len(df_filtered) / len(df) * 100) if len(df) > 0 else 0

print(f'  Before: {len(df):,} rows')
print(f'  After PSC check: {len(df_filtered):,} rows ({psc_pct:.1f}%)')
print(f'  Removed: {psc_removed:,} rows')

if psc_removed > len(df) * 0.1:
    print('  ⚠ WARNING: PSC filter removed >10% of rows. Verify download filters.')

run_metadata['total_rows_after_psc_check'] = len(df_filtered)
run_metadata['total_rows_after_value_check'] = len(df_filtered)

# Save filtered checkpoint
df_filtered.to_csv(INTERIM_OUTPUT, index=False)
print(f'\n✓ Saved filtered checkpoint: {INTERIM_OUTPUT}')

NameError: df is not defined. Run Cell 9 (Data Loading and Schema Discovery) first.

## 6. PIID-Group Sampling

Sample complete PIID groups to preserve contract modification histories. Set SAMPLE_CONTRACTS = None to use all data.

In [ ]:
# Verify required data and config
if 'df_filtered' not in globals():
    raise NameError('df_filtered not defined. Run Filtering cell first.')

if 'COLUMN_MAP' not in globals() or 'piid' not in COLUMN_MAP:
    raise KeyError('COLUMN_MAP["piid"] not defined. Run Configuration cell first.')

if 'SAMPLE_CONTRACTS' not in globals():
    SAMPLE_CONTRACTS = None

if 'RANDOM_STATE' not in globals():
    RANDOM_STATE = 42

if 'np' not in globals():
    import numpy as np

if 'run_metadata' not in globals():
    run_metadata = {}

# Get unique PIIDs
piid_col = COLUMN_MAP['piid']
unique_piids = df_filtered[piid_col].unique()
total_unique_piids = len(unique_piids)

print(f'Total unique contracts (PIIDs): {total_unique_piids:,}')

run_metadata['unique_piids_before_sample'] = total_unique_piids

# Determine if sampling is needed
if SAMPLE_CONTRACTS is None or total_unique_piids <= SAMPLE_CONTRACTS:
    # Use all PIIDs
    sample_df = df_filtered.copy()
    actual_sample_size = total_unique_piids
    print(f'Using all {total_unique_piids:,} PIIDs (no sampling)')
else:
    # Sample PIIDs
    print(f'Target sample size: {SAMPLE_CONTRACTS:,}')
    rng = np.random.RandomState(RANDOM_STATE)
    sampled_piids = rng.choice(unique_piids, size=SAMPLE_CONTRACTS, replace=False)
    sample_df = df_filtered[df_filtered[piid_col].isin(sampled_piids)].copy()
    actual_sample_size = SAMPLE_CONTRACTS
    print(f'  Sampled {SAMPLE_CONTRACTS:,} PIIDs → {len(sample_df):,} total rows')

run_metadata['unique_piids_after_sample'] = actual_sample_size

# Reset index
sample_df = sample_df.reset_index(drop=True)
print(f'\nWorking sample shape: {sample_df.shape}')

Total unique contracts (PIIDs): 46,152
Using all 46,152 PIIDs (no sampling)

Working sample shape: (230088, 18)


## 7. Outcome Label Construction

Group by PIID to construct binary labels for cost overruns (`over_budget`) and schedule delays (`late`).

In [ ]:
# Verify required data
if 'sample_df' not in globals():
    raise NameError('sample_df not defined. Run PIID Sampling cell first.')

if 'COLUMN_MAP' not in globals():
    raise KeyError('COLUMN_MAP not defined. Run Configuration cell first.')

if 'compute_cost_growth' not in globals() or 'compute_delay' not in globals():
    raise NameError('Helper functions not defined. Run Helper Functions cell first.')

if 'tqdm' not in globals():
    from tqdm import tqdm

if 'MIN_CONTRACT_VALUE' not in globals():
    MIN_CONTRACT_VALUE = 500_000

if 'run_metadata' not in globals():
    run_metadata = {}

# Type-cast dollar columns
for col in [COLUMN_MAP['base_all_options'], COLUMN_MAP['base_exercised']]:
    sample_df[col] = pd.to_numeric(
        sample_df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )

# Type-cast date columns
for col in [COLUMN_MAP['current_completion'], COLUMN_MAP['ultimate_completion'],
            COLUMN_MAP['effective_date'], COLUMN_MAP['signed_date']]:
    sample_df[col] = pd.to_datetime(sample_df[col], errors='coerce')

# Sort by PIID and modification number
piid_col = COLUMN_MAP['piid']
mod_col = COLUMN_MAP['mod_number']
sample_df = sample_df.sort_values([piid_col, mod_col])

print('Grouping by PIID to construct labels...')
label_rows = []
piids_filtered = 0

# Iterate through each PIID group
for piid, group in tqdm(sample_df.groupby(piid_col), total=sample_df[piid_col].nunique(), desc='Processing PIIDs'):
    # Get first (initial) and last (final) modification
    first = group.iloc[0]
    last = group.iloc[-1]
    
    # Extract cost values
    base_val = first[COLUMN_MAP['base_all_options']]
    final_val = last[COLUMN_MAP['base_exercised']]
    
    # PIID-group level value filter: skip if initial contract value < MIN_CONTRACT_VALUE
    if pd.notna(base_val) and base_val >= MIN_CONTRACT_VALUE:
        # Extract dates
        current_date = first[COLUMN_MAP['current_completion']]
        ultimate_date = last[COLUMN_MAP['ultimate_completion']]
        
        # Compute derived metrics
        cost_growth = compute_cost_growth(base_val, final_val)
        delay = compute_delay(current_date, ultimate_date)
        
        # Build label row
        row = {
            'piid': piid,
            'description': first[COLUMN_MAP['description']],
            'psc': first[COLUMN_MAP['psc']],
            'naics': first[COLUMN_MAP['naics']],
            'contract_type': first[COLUMN_MAP['contract_type']],
            'extent_competed': first[COLUMN_MAP['extent_competed']],
            'num_offers': first[COLUMN_MAP['num_offers']],
            'agency_id': first[COLUMN_MAP['agency_id']],
            'vendor_name': first[COLUMN_MAP['vendor_name']],
            'base_value': base_val,
            'final_value': final_val,
            'cost_growth_pct': cost_growth * 100 if pd.notna(cost_growth) else np.nan,
            'delay_days': delay,
            'num_modifications': len(group),
        }
        
        if 'state_code' in COLUMN_MAP:
            row['state_code'] = first[COLUMN_MAP['state_code']]
        
        label_rows.append(row)
    else:
        piids_filtered += 1

# Create labeled DataFrame
labeled_df = pd.DataFrame(label_rows)

print(f'\n✓ Labeled DataFrame created: {len(labeled_df):,} rows')
print(f'  PIIDs filtered (value < {MIN_CONTRACT_VALUE:,}): {piids_filtered:,}')

Grouping by PIID to construct labels...


Processing PIIDs: 100%|██████████| 46152/46152 [00:13<00:00, 3421.21it/s]



Labeled DataFrame created: 45,456 rows
  PIIDs filtered by value: 696 (base_value < 500,000)


### 7.1 Apply Binary Labels and Adaptive Threshold

In [ ]:
# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run Label Construction cell first.')

if 'COST_OVERRUN_THRESHOLD' not in globals() or 'COST_OVERRUN_THRESHOLD_FALLBACK' not in globals():
    COST_OVERRUN_THRESHOLD = 0.05
    COST_OVERRUN_THRESHOLD_FALLBACK = 0.01

if 'SCHEDULE_DELAY_THRESHOLD' not in globals():
    SCHEDULE_DELAY_THRESHOLD = 0

if 'run_metadata' not in globals():
    run_metadata = {}

# Apply initial cost overrun threshold (5%)
threshold = COST_OVERRUN_THRESHOLD
labeled_df['over_budget'] = (labeled_df['cost_growth_pct'] > threshold * 100).astype(int)

# Adaptive threshold: if positive class is too small (<5%), drop to 1%
positive_rate = labeled_df['over_budget'].mean()
if positive_rate < 0.05:
    threshold = COST_OVERRUN_THRESHOLD_FALLBACK
    labeled_df['over_budget'] = (labeled_df['cost_growth_pct'] > threshold * 100).astype(int)
    print(f'Adaptive threshold applied: {threshold:.0%}')
    print(f'  (original 5% yielded only {positive_rate:.2%} positives)')
else:
    print(f'Using standard threshold: {threshold:.0%}')

# Schedule delay label (any positive delay = late)
labeled_df['late'] = (labeled_df['delay_days'] > SCHEDULE_DELAY_THRESHOLD).astype(int)

# Drop rows with missing labels
before_drop = len(labeled_df)
labeled_df = labeled_df.dropna(subset=['cost_growth_pct', 'delay_days']).copy()
after_drop = len(labeled_df)
valid_pct = (after_drop / before_drop * 100) if before_drop > 0 else 0

print(f'\nValid labels: {after_drop:,} / {before_drop:,} ({valid_pct:.1f}%)')

# Print class balance
print('\n' + '='*50)
print('CLASS BALANCE SUMMARY')
print('='*50)
total = len(labeled_df)
if total == 0:
    print('  ⚠ No labeled rows available after filtering.')
else:
    for target in ['over_budget', 'late']:
        pos = labeled_df[target].sum()
        neg = total - pos
        print(f'\n{target}:')
        print(f'  Positive: {pos:6,} ({pos/total*100:5.2f}%)')
        print(f'  Negative: {neg:6,} ({neg/total*100:5.2f}%)')

run_metadata['final_labeled_count'] = after_drop
run_metadata['over_budget_positives'] = int(labeled_df['over_budget'].sum())
run_metadata['late_positives'] = int(labeled_df['late'].sum())

Using standard threshold: 5%

Valid labels: 45,456 / 45,456 (100.0%)

Class Balance Summary:
----------------------------------------
  over_budget :  4,725 positive (10.39%) | 40,731 negative (89.61%)
  late        : 27,961 positive (61.51%) | 17,495 negative (38.49%)


### 7.2 Additional Derived Features

In [ ]:
# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run Binary Labels cell first.')

if 'np' not in globals():
    import numpy as np

if 'pd' not in globals():
    import pandas as pd

print('Creating derived features...')

# Log-transformed initial cost
labeled_df['log_initial_cost'] = np.log1p(labeled_df['base_value'].fillna(0))

# Clean num_offers
labeled_df['num_offers'] = pd.to_numeric(labeled_df['num_offers'], errors='coerce').fillna(0)

# Value tier bins
labeled_df['value_tier'] = pd.cut(
    labeled_df['base_value'],
    bins=[0, 1e6, 5e6, 25e6, 100e6, float('inf')],
    labels=['<1M', '1M-5M', '5M-25M', '25M-100M', '>100M']
)

print('✓ Derived features created:')
print('  - log_initial_cost')
print('  - num_offers (cleaned)')
print('  - value_tier (categorical)')

Derived features created:
  - log_initial_cost
  - num_offers (cleaned numeric)
  - value_tier (categorical bins)


## 8. Save Outputs and Generate Summary

In [ ]:
# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run all previous cells first.')

if 'FINAL_OUTPUT' not in globals():
    from pathlib import Path
    _data_dir = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    FINAL_OUTPUT = str(_data_dir / 'processed' / 'labeled_contracts.csv')

# Save final labeled dataset
labeled_df.to_csv(FINAL_OUTPUT, index=False)
print(f'✓ Saved {len(labeled_df):,} labeled contracts')
print(f'  {FINAL_OUTPUT}')

Saved 45,456 labeled contracts to:
   ../data/processed/labeled_contracts.csv


### 8.1 Generate Dataset Summary Report

In [ ]:
# Build summary report
def _pct(numerator, denominator):
    return (numerator / denominator * 100) if denominator else 0.0

# Verify required data
if 'run_metadata' not in globals():
    raise NameError('run_metadata not defined. Run all previous cells first.')

if 'SUMMARY_FILE' not in globals() or 'INTERIM_OUTPUT' not in globals() or 'FINAL_OUTPUT' not in globals():
    from pathlib import Path
    _data_dir = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    INTERIM_OUTPUT = globals().get('INTERIM_OUTPUT', str(_data_dir / 'interim' / 'filtered_physical_deliverables.csv'))
    FINAL_OUTPUT = globals().get('FINAL_OUTPUT', str(_data_dir / 'processed' / 'labeled_contracts.csv'))
    SUMMARY_FILE = globals().get('SUMMARY_FILE', str(_data_dir / 'processed' / 'dataset_summary.txt'))

# Extract metadata
total_loaded = run_metadata.get('total_rows_loaded', 0)
psc_kept = run_metadata.get('total_rows_after_psc_check', 0)
value_kept = run_metadata.get('total_rows_after_value_check', 0)
final_count = run_metadata.get('final_labeled_count', 0)
over_pos = run_metadata.get('over_budget_positives', 0)
late_pos = run_metadata.get('late_positives', 0)
run_date = run_metadata.get('run_date', 'N/A')

# Build report
summary_lines = [
    '=' * 70,
    'DATASET SUMMARY REPORT - USAspending Parquet Files',
    '=' * 70,
    f"Run Date: {run_date}",
    f"Data Source: C:/Users/sarme/IS392Final/data",
    '',
    '-' * 70,
    'DATA LOADING',
    '-' * 70,
    f"Total rows loaded: {total_loaded:,}",
    '',
    '-' * 70,
    'FILTERING SANITY CHECKS',
    '-' * 70,
    f"After PSC check: {psc_kept:,}",
    f"After value check: {value_kept:,}",
    f"PSC retention rate: {_pct(psc_kept, total_loaded):.2f}%",
    '',
    '-' * 70,
    'SAMPLING',
    '-' * 70,
    f"Unique PIIDs before sampling: {run_metadata.get('unique_piids_before_sample', 0):,}",
    f"Sampled PIIDs: {run_metadata.get('unique_piids_after_sample', 0):,}",
    '',
    '-' * 70,
    'LABEL DISTRIBUTION',
    '-' * 70,
    f"Final labeled contracts: {final_count:,}",
    '',
    f"over_budget: {over_pos:,} positive ({_pct(over_pos, final_count):.2f}%)",
    f"             {final_count - over_pos:,} negative ({_pct(final_count - over_pos, final_count):.2f}%)",
    '',
    f"late:        {late_pos:,} positive ({_pct(late_pos, final_count):.2f}%)",
    f"             {final_count - late_pos:,} negative ({_pct(final_count - late_pos, final_count):.2f}%)",
    '',
    '-' * 70,
    'OUTPUT FILES',
    '-' * 70,
    f"Intermediate: {INTERIM_OUTPUT}",
    f"Final:        {FINAL_OUTPUT}",
    '=' * 70,
]

# Write summary
summary_text = '\n'.join(summary_lines)
with open(SUMMARY_FILE, 'w', encoding='utf-8') as f:
    f.write(summary_text)

# Print summary
print(summary_text)
print(f'\n✓ Summary saved: {SUMMARY_FILE}')

DATASET SUMMARY REPORT - USAspending CSV
Run Date: 2026-04-24 14:18:44

----------------------------------------------------------------------
DATA LOADING
----------------------------------------------------------------------
Total rows loaded: 233,642

----------------------------------------------------------------------
FILTERING SANITY CHECKS
----------------------------------------------------------------------
After PSC check: 230,088
After value check: 230,088
PSC retention rate: 98.48%

----------------------------------------------------------------------
SAMPLING
----------------------------------------------------------------------
Unique PIIDs before sampling: 46,152
Sampled PIIDs: 46,152

----------------------------------------------------------------------
LABEL DISTRIBUTION
----------------------------------------------------------------------
Final labeled contracts: 45,456

over_budget: 4,725 positive (10.39%)
             40,731 negative (89.61%)

late:        27,96

## 9. Next Steps

This notebook produces the foundation dataset for all downstream analysis. Next:

1. **Run `05_text_preprocessing_lda.ipynb`** — Apply two-track text preprocessing
2. **Run `06_classification.ipynb`** — Build feature matrices, train classification models
3. **(Optional) Run `07_gao_validation.ipynb`** — Cross-validate with GAO data

**Key outputs:**
- `data/processed/labeled_contracts.csv` — Complete labeled dataset
- `data/processed/dataset_summary.txt` — Metadata and class balance summary